In [1]:
!pip install pdfplumber requests beautifulsoup4 pandas
!pip install pdf2image pytesseract
!apt-get install -y tesseract-ocr tesseract-ocr-por poppler-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 26.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 39.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 24.4 MB/s  0:00:00
  Attempting uninstall: Pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [pdfplumber]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pdf2image]
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (5.3.4-1build5).
The following NEW packages will be installed:
  poppler-utils tesseract-ocr-p

In [2]:

from pathlib import Path
import hashlib
import json
import re
import shutil
import pandas as pd
import pdfplumber
from bs4 import BeautifulSoup
import os
from datetime import datetime
from google.colab import drive


print("Bibliotecas importadas.")

Bibliotecas importadas.


In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
raiz = Path("Corpus_Saneamento")

pastas = [
    "dados_brutos/resolucoes_reguladora",
    "dados_brutos/manuais_sabesp",
    "dados_brutos/atendimentos_sinteticos",
    "dados_processados",
    "metadados",
    "scripts",
    "logs",
    "resultados"
]

for pasta in pastas:
    (raiz / pasta).mkdir(parents=True, exist_ok=True)

print("✅ Estrutura de pastas criada:")
for caminho in sorted(raiz.rglob("*")):
    if caminho.is_dir():
        print(f"  {caminho}")

✅ Estrutura de pastas criada:
  Corpus_Saneamento/dados_brutos
  Corpus_Saneamento/dados_brutos/atendimentos_sinteticos
  Corpus_Saneamento/dados_brutos/manuais_sabesp
  Corpus_Saneamento/dados_brutos/resolucoes_reguladora
  Corpus_Saneamento/dados_processados
  Corpus_Saneamento/logs
  Corpus_Saneamento/metadados
  Corpus_Saneamento/resultados
  Corpus_Saneamento/scripts


In [5]:
def limpar_texto(texto):
    """Remove espaços extras, quebras de linha desnecessárias e caracteres estranhos."""
    texto = re.sub(r'\n+', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto)
    texto = re.sub(r'[^\w\s.,;:()\\/\-]', '', texto)
    return texto.strip()


def extrair_pdf(caminho_pdf):
    """Extrai e limpa o texto de todas as páginas de um PDF."""
    texto_completo = []
    num_paginas = 0

    with pdfplumber.open(caminho_pdf) as pdf:
        num_paginas = len(pdf.pages)
        for pagina in pdf.pages:
            texto = pagina.extract_text()
            if texto:
                texto_completo.append(limpar_texto(texto))

    return " ".join(texto_completo), num_paginas


def extrair_pdf_ocr(caminho_pdf):
    """Extrai texto de PDFs baseados em imagem usando OCR."""
    from pdf2image import convert_from_path
    import pytesseract
    paginas = convert_from_path(caminho_pdf, dpi=200)
    texto_completo = []
    for i, pagina in enumerate(paginas):
        texto = pytesseract.image_to_string(pagina, lang="por")
        if texto.strip():
            texto_completo.append(limpar_texto(texto))
        print(f"  Página {i+1}: {len(texto)} caracteres extraídos via OCR")
    return " ".join(texto_completo), len(paginas)


def processar_pasta(pasta_origem, categoria, raiz):
    pasta_origem = Path(pasta_origem)
    pasta_processados = raiz / "dados_processados"
    pasta_metadados = raiz / "metadados"

    pdfs = list(pasta_origem.glob("*.pdf"))
    print(f"\n📂 Pasta: {pasta_origem.name} | PDFs encontrados: {len(pdfs)}")

    registros = []

    for pdf in pdfs:
        print(f"  Lendo: {pdf.name}...", end=" ")
        try:
            texto, num_paginas = extrair_pdf(pdf)
            if not texto.strip():
                print("⚠️ Sem texto extraível — tentando OCR...")
                texto, num_paginas = extrair_pdf_ocr(pdf)
            if not texto.strip():
                print("❌ Sem texto mesmo via OCR.")
                continue
            nome_saida = f"{categoria}_{pdf.stem}.txt"
            caminho_saida = pasta_processados / nome_saida
            caminho_saida.write_text(texto, encoding="utf-8")
            registro = {
                "arquivo_original": pdf.name,
                "arquivo_processado": nome_saida,
                "categoria": categoria,
                "num_paginas": num_paginas,
                "num_caracteres": len(texto),
                "num_palavras": len(texto.split()),
                "data_coleta": datetime.now().strftime("%Y-%m-%d"),
                "fonte": str(pasta_origem)
            }
            registros.append(registro)
            print(f"✅ {num_paginas} páginas | {len(texto.split())} palavras")
        except Exception as e:
            print(f"❌ Erro: {e}")

    if registros:
        caminho_meta = pasta_metadados / f"metadados_{categoria}.json"
        with open(caminho_meta, "w", encoding="utf-8") as f:
            json.dump(registros, f, ensure_ascii=False, indent=2)
        print(f"  💾 Metadados salvos em: {caminho_meta.name}")

    return registros

print("✅ Funções base definidas!")

✅ Funções base definidas!


In [6]:
# ── CÉLULA 6-B: Integrar corpus sintético ao pipeline ─────────────────────
import json
from pathlib import Path

# Faça upload do arquivo corpus_sintetico_saae.jsonl para o Drive antes de executar,
# ou use o caminho direto se já estiver na pasta do projeto.
# Opção 1: arquivo na raiz do Drive
caminho_jsonl = Path("/content/drive/MyDrive/Corpora_TCC/Arquivo/corpus_sintetico_saae.jsonl")

# Opção 2: se você colocar dentro da pasta do TCC
# caminho_jsonl = drive_base / "corpus_sintetico_saae.jsonl"

pasta_processados = raiz / "dados_processados"
pasta_processados.mkdir(parents=True, exist_ok=True)

with open(caminho_jsonl, encoding="utf-8") as f:
    entradas = [json.loads(linha) for linha in f if linha.strip()]

print(f"📂 Corpus sintético | Entradas encontradas: {len(entradas)}")
contagem = 0
for entrada in entradas:
    id_entrada = entrada.get("id", f"sint_{contagem:03d}")
    conteudo = entrada.get("conteudo", "").strip()
    if not conteudo:
        print(f"  ⚠️  {id_entrada}: conteúdo vazio, ignorado.")
        continue
    # Cabeçalho com metadados para ajudar o RAG a identificar o contexto
    categoria = entrada.get("categoria", "geral")
    subcategoria = entrada.get("subcategoria", "")
    cabecalho = f"[CORPUS SINTÉTICO | {categoria} | {subcategoria}]\n"
    texto_final = cabecalho + conteudo
    caminho_saida = pasta_processados / f"{id_entrada}.txt"
    caminho_saida.write_text(texto_final, encoding="utf-8")
    contagem += 1
    print(f"  ✅ {id_entrada} → {caminho_saida.name} ({len(conteudo.split())} palavras)")

print(f"\n💾 {contagem} arquivo(s) sintético(s) gravados em dados_processados/")
print("▶️  Agora re-execute a Célula 14 para reindexar o ChromaDB com todos os documentos.")

📂 Corpus sintético | Entradas encontradas: 11
  ✅ saae_sint_001 → saae_sint_001.txt (163 palavras)
  ✅ saae_sint_002 → saae_sint_002.txt (134 palavras)
  ✅ saae_sint_003 → saae_sint_003.txt (125 palavras)
  ✅ saae_sint_004 → saae_sint_004.txt (140 palavras)
  ✅ saae_sint_005 → saae_sint_005.txt (142 palavras)
  ✅ saae_sint_006 → saae_sint_006.txt (87 palavras)
  ✅ saae_sint_007 → saae_sint_007.txt (146 palavras)
  ✅ saae_sint_008 → saae_sint_008.txt (119 palavras)
  ✅ saae_sint_009 → saae_sint_009.txt (71 palavras)
  ✅ saae_sint_010 → saae_sint_010.txt (104 palavras)
  ✅ saae_sint_011 → saae_sint_011.txt (170 palavras)

💾 11 arquivo(s) sintético(s) gravados em dados_processados/
▶️  Agora re-execute a Célula 14 para reindexar o ChromaDB com todos os documentos.


In [7]:
import hashlib

PADROES_PII = [
    (r'\(?\d{2}\)?\s?\d{4,5}[-\s]?\d{4}', '[TELEFONE]'),
    (r'\b20\d{12}\b', '[PROTOCOLO]'),
    (r'[\w\.-]+@[\w\.-]+\.\w+', '[EMAIL]'),
    (r'\d{3}[\.\s]?\d{3}[\.\s]?\d{3}[-\s]?\d{2}', '[CPF]'),
]

PADROES_LINHA_PII = [
    r'^Com\s+\w+',
    r'meu nome é',
    r'Falo com\s+\w+',
    r'Início do chamado\s+-\s+Atendimento',
    r'Chamado transferido',
    r'Fim do chamado',
    r'Chamado número:',
    r'Início do chamado:',
    r'Término do chamado:',
]

def anonimizar_historico(texto):
    linhas = texto.split('\n')
    linhas_limpas = []
    for linha in linhas:
        ignorar = False
        for padrao in PADROES_LINHA_PII:
            if re.search(padrao, linha, re.IGNORECASE):
                ignorar = True
                break
        if ignorar:
            continue
        for padrao, substituto in PADROES_PII:
            linha = re.sub(padrao, substituto, linha)
        linhas_limpas.append(linha)
    return '\n'.join(linhas_limpas)

def anonimizar_nome_arquivo(nome_pdf):
    hash_id = hashlib.md5(nome_pdf.encode()).hexdigest()[:8]
    return f"historico_{hash_id}"

def processar_historicos_atendimento(pasta_origem, raiz):
    pasta_origem = Path(pasta_origem)
    pasta_processados = raiz / "dados_processados"
    pasta_metadados = raiz / "metadados"

    pdfs = list(pasta_origem.glob("*.pdf"))
    print(f"\n📂 Históricos de atendimento | PDFs encontrados: {len(pdfs)}")

    registros = []

    for pdf in pdfs:
        id_anonimo = anonimizar_nome_arquivo(pdf.name)
        print(f"  Lendo: {id_anonimo}...", end=" ")
        try:
            # Extrai SEM limpar_texto para preservar as quebras de linha
            texto_bruto = []
            num_paginas = 0
            with pdfplumber.open(pdf) as doc:
                num_paginas = len(doc.pages)
                for pagina in doc.pages:
                    t = pagina.extract_text()
                    if t:
                        texto_bruto.append(t)
            texto_bruto = "\n".join(texto_bruto)

            if not texto_bruto.strip():
                print("⚠️ Sem texto — tentando OCR...")
                texto_bruto, num_paginas = extrair_pdf_ocr(pdf)

            if not texto_bruto.strip():
                print("❌ Sem texto mesmo via OCR.")
                continue

            # Anonimiza preservando estrutura de linhas
            texto_anonimizado = anonimizar_historico(texto_bruto)

            # Limpa espaços excessivos mas mantém parágrafos
            texto_anonimizado = re.sub(r'\n{3,}', '\n\n', texto_anonimizado)
            texto_anonimizado = re.sub(r'[ \t]+', ' ', texto_anonimizado).strip()

            if not texto_anonimizado.strip():
                print("⚠️ Texto vazio após anonimização — verificar padrões.")
                continue

            nome_saida = f"{id_anonimo}.txt"
            caminho_saida = pasta_processados / nome_saida
            caminho_saida.write_text(texto_anonimizado, encoding="utf-8")

            registros.append({
                "arquivo_original": "[ANONIMIZADO]",
                "arquivo_processado": nome_saida,
                "categoria": "historico_atendimento",
                "num_paginas": num_paginas,
                "num_caracteres": len(texto_anonimizado),
                "num_palavras": len(texto_anonimizado.split()),
                "data_coleta": datetime.now().strftime("%Y-%m-%d"),
                "fonte": "Historico_Atendimento/SAAE_Sao_Carlos",
                "anonimizado": True
            })
            print(f"✅ {num_paginas} pág. | {len(texto_anonimizado.split())} palavras")

        except Exception as e:
            print(f"❌ Erro: {e}")

    if registros:
        caminho_meta = pasta_metadados / "metadados_historicos.json"
        with open(caminho_meta, "w", encoding="utf-8") as f:
            json.dump(registros, f, ensure_ascii=False, indent=2)
        print(f"  💾 {len(registros)} arquivo(s) processados e anonimizados.")

    return registros

print("✅ Funções de histórico definidas!")


✅ Funções de histórico definidas!


In [8]:
# Caminhos das pastas no seu Google Drive
drive_base = Path("/content/drive/MyDrive/Corpora_TCC/Arquivo")

todos_registros = []

todos_registros += processar_pasta(
    pasta_origem=drive_base / "Ares_PCJ",
    categoria="resolucao",
    raiz=raiz
)

todos_registros += processar_pasta(
    pasta_origem=drive_base / "Sabesp",
    categoria="manual",
    raiz=raiz
)

todos_registros += processar_historicos_atendimento(
    pasta_origem=drive_base / "Historico_Atendimento",
    raiz=raiz
)

# Gera CSV consolidado com todos os metadados
if todos_registros:
    df_meta = pd.DataFrame(todos_registros)
    caminho_csv = raiz / "metadados" / "metadados_geral.csv"
    df_meta.to_csv(caminho_csv, index=False, encoding="utf-8")
    print("\n" + "="*50)
    print("✅ PROCESSAMENTO CONCLUÍDO")
    print("="*50)
    print(df_meta[["arquivo_processado", "categoria", "num_paginas", "num_palavras"]].to_string(index=False))


📂 Pasta: Ares_PCJ | PDFs encontrados: 2
  Lendo: 1787764462-resoluo_n_724_2026_-_reajuste_tarifrio_-_so_carlos.pdf... ✅ 8 páginas | 2452 palavras
  Lendo: 1678300883-resolucao_n_482_2023_-_regulamento_saae_-_sao_carlos (1).pdf... ✅ 46 páginas | 14076 palavras
  💾 Metadados salvos em: metadados_resolucao.json

📂 Pasta: Sabesp | PDFs encontrados: 2
  Lendo: folheto-manual-instalacao-hidraulica.pdf... ✅ 9 páginas | 1402 palavras
  Lendo: folheto-variacao-consumo.pdf... ⚠️ Sem texto extraível — tentando OCR...
  Página 1: 584 caracteres extraídos via OCR
  Página 2: 735 caracteres extraídos via OCR
  Página 3: 690 caracteres extraídos via OCR
  Página 4: 689 caracteres extraídos via OCR
  Página 5: 902 caracteres extraídos via OCR
  Página 6: 193 caracteres extraídos via OCR
  Página 7: 324 caracteres extraídos via OCR
  Página 8: 158 caracteres extraídos via OCR
✅ 8 páginas | 696 palavras
  💾 Metadados salvos em: metadados_manual.json

📂 Históricos de atendimento | PDFs encontrados: 33
 

✅ 5 pág. | 680 palavras
  Lendo: historico_a8f0260f... 

✅ 4 pág. | 396 palavras
  Lendo: historico_183998f2... 

✅ 4 pág. | 542 palavras
  Lendo: historico_39920724... 

✅ 7 pág. | 645 palavras
  Lendo: historico_101c7558... 

✅ 3 pág. | 345 palavras
  Lendo: historico_d8adea23... 

... (24 outputs left out)

✅ 3 pág. | 299 palavras
  Lendo: historico_c66cac7d... 

✅ 3 pág. | 304 palavras
  Lendo: historico_2ef939b4... 

✅ 4 pág. | 397 palavras
  Lendo: historico_94c9983c... 

✅ 4 pág. | 302 palavras
  💾 33 arquivo(s) processados e anonimizados.

✅ PROCESSAMENTO CONCLUÍDO
                                                               arquivo_processado             categoria  num_paginas  num_palavras
      resolucao_1787764462-resoluo_n_724_2026_-_reajuste_tarifrio_-_so_carlos.txt             resolucao            8          2452
resolucao_1678300883-resolucao_n_482_2023_-_regulamento_saae_-_sao_carlos (1).txt             resolucao           46         14076
                                  manual_folheto-manual-instalacao-hidraulica.txt                manual            9          1402
                                              manual_folheto-variacao-consumo.txt                manual            8           696
                                                           historico_3a8a7c7c.txt historico_atendimento            5           680
                                                           historico_a8f0260f.txt historico_atendimento            4 

In [9]:
import shutil

# Copia os resultados gerados no Colab de volta para o Drive
destino_drive = Path("/content/drive/MyDrive/Corpora_TCC/resultados_processamento")
destino_drive.mkdir(parents=True, exist_ok=True)

shutil.copytree(
    raiz / "dados_processados",
    destino_drive / "dados_processados",
    dirs_exist_ok=True
)

shutil.copytree(
    raiz / "metadados",
    destino_drive / "metadados",
    dirs_exist_ok=True
)

print("✅ Resultados salvos no Google Drive em:")
print(f"   {destino_drive}")

✅ Resultados salvos no Google Drive em:
   /content/drive/MyDrive/Corpora_TCC/resultados_processamento


In [10]:
from pdf2image import convert_from_path
import pytesseract

def extrair_pdf_ocr(caminho_pdf):
    """Extrai texto de PDFs baseados em imagem usando OCR."""
    print(f"  Convertendo páginas para imagem...")
    paginas = convert_from_path(caminho_pdf, dpi=200)

    texto_completo = []
    for i, pagina in enumerate(paginas):
        texto = pytesseract.image_to_string(pagina, lang="por")
        if texto.strip():
            texto_completo.append(limpar_texto(texto))
        print(f"  Página {i+1}: {len(texto)} caracteres extraídos via OCR")

    return " ".join(texto_completo), len(paginas)

# Testa no arquivo problemático
pdf = drive_base / "Sabesp" / "folheto-variacao-consumo.pdf"
texto, paginas = extrair_pdf_ocr(pdf)

print(f"\nTotal de palavras extraídas: {len(texto.split())}")
print(f"\nPrimeiros 500 caracteres:\n{texto[:500]}")

  Convertendo páginas para imagem...
  Página 1: 584 caracteres extraídos via OCR
  Página 2: 735 caracteres extraídos via OCR
  Página 3: 690 caracteres extraídos via OCR
  Página 4: 689 caracteres extraídos via OCR
  Página 5: 902 caracteres extraídos via OCR
  Página 6: 193 caracteres extraídos via OCR
  Página 7: 324 caracteres extraídos via OCR
  Página 8: 158 caracteres extraídos via OCR

Total de palavras extraídas: 696

Primeiros 500 caracteres:
1 sabesp Início  Serviços  Para Você  Variações de Consumo Variações de Consumo Se a conta de água está mais alta que o normal, isso pode indicar aumento de consumo ou algum problema nas instalações internas. É algo que acontece com frequência e pode ter diversas explicações simples. As principais causas para um consumo elevado podem ser: Cobrança de serviços de meses anteriores; Aumento do número de pessoas no imóvel; Ceder água para terceiros; Uso da água para construção ou reforma; Limpeza de


In [11]:
# Extrai via OCR
pdf = drive_base / "Sabesp" / "folheto-variacao-consumo.pdf"
texto, num_paginas = extrair_pdf_ocr(pdf)

if texto.strip():
    # Salva localmente no Colab
    nome_saida = "manual_folheto-variacao-consumo.txt"
    caminho_saida = raiz / "dados_processados" / nome_saida
    caminho_saida.write_text(texto, encoding="utf-8")
    print(f"✅ Salvo localmente: {caminho_saida}")

    # Salva no Google Drive
    caminho_drive = Path("/content/drive/MyDrive/Corpora_TCC/resultados_processamento/dados_processados") / nome_saida
    caminho_drive.write_text(texto, encoding="utf-8")
    print(f"✅ Salvo no Drive: {caminho_drive}")

    # Adiciona aos metadados existentes
    registro_ocr = {
        "arquivo_original": "folheto-variacao-consumo.pdf",
        "arquivo_processado": nome_saida,
        "categoria": "manual",
        "num_paginas": num_paginas,
        "num_caracteres": len(texto),
        "num_palavras": len(texto.split()),
        "data_coleta": datetime.now().strftime("%Y-%m-%d"),
        "fonte": str(drive_base / "Sabesp"),
        "metodo_extracao": "OCR"
    }

    # Atualiza o CSV de metadados
    caminho_csv = raiz / "metadados" / "metadados_geral.csv"
    df_meta = pd.read_csv(caminho_csv)
    df_novo = pd.DataFrame([registro_ocr])
    df_meta = pd.concat([df_meta, df_novo], ignore_index=True)
    df_meta.to_csv(caminho_csv, index=False, encoding="utf-8")

    # Salva CSV atualizado no Drive também
    caminho_csv_drive = Path("/content/drive/MyDrive/Corpora_TCC/resultados_processamento/metadados/metadados_geral.csv")
    df_meta.to_csv(caminho_csv_drive, index=False, encoding="utf-8")

    print(f"\n📊 Metadados atualizados. Total de documentos: {len(df_meta)}")
    print(df_meta[["arquivo_original", "categoria", "num_paginas", "num_palavras", "metodo_extracao"]].to_string(index=False))
else:
    print("❌ OCR não conseguiu extrair texto deste arquivo.")

  Convertendo páginas para imagem...
  Página 1: 584 caracteres extraídos via OCR
  Página 2: 735 caracteres extraídos via OCR
  Página 3: 690 caracteres extraídos via OCR
  Página 4: 689 caracteres extraídos via OCR
  Página 5: 902 caracteres extraídos via OCR
  Página 6: 193 caracteres extraídos via OCR
  Página 7: 324 caracteres extraídos via OCR
  Página 8: 158 caracteres extraídos via OCR
✅ Salvo localmente: Corpus_Saneamento/dados_processados/manual_folheto-variacao-consumo.txt
✅ Salvo no Drive: /content/drive/MyDrive/Corpora_TCC/resultados_processamento/dados_processados/manual_folheto-variacao-consumo.txt

📊 Metadados atualizados. Total de documentos: 38
                                                       arquivo_original             categoria  num_paginas  num_palavras metodo_extracao
      1787764462-resoluo_n_724_2026_-_reajuste_tarifrio_-_so_carlos.pdf             resolucao            8          2452             NaN
1678300883-resolucao_n_482_2023_-_regulamento_saae_-_sa

In [12]:
!pip install sentence-transformers chromadb groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 57.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 48.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 47.4 MB/s  0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.42.1
    Uninstalling opentelemetry-api-1.42.1:
      Successfully uninstalled opentelemetry-api-1.42.1
  Attempting uninstall: oauthlib
    Found existing installation: oauthlib 3.3.1
    Uninstalling oauthlib-3.3.1:
      Successfully uninstalled oauthlib-3.3.1
  Attempting uninstall: opentelemetry-semantic-conventions
    Found existing installation: opentelemetry-semantic-conventions 0.63b1
    Uninstalling opentelemetry-semantic-conventions-0.63b1:
      Successfully uninstalled opentelemetry-semantic-conventions

In [13]:
from sentence_transformers import SentenceTransformer
import chromadb
from groq import Groq
from pathlib import Path
import os

In [14]:
!pip install --upgrade pillow

Após executar a célula acima para atualizar o `pillow`, pode ser necessário reiniciar o ambiente de execução caso o erro persista. Para reiniciar, clique em **Ambiente de execução (Runtime) -> Reiniciar sessão (Restart session)** e depois execute as células de importação novamente.

In [15]:
!pip install -q google-generativeai

import google.generativeai as genai
from sentence_transformers import SentenceTransformer
import chromadb
from pathlib import Path
import re

# ── 1. Carrega o modelo de embeddings ──────────────────────
print("⏳ Carregando modelo de embeddings...")
modelo_embeddings = SentenceTransformer("neuralmind/bert-base-portuguese-cased")
print(f"✅ Modelo carregado! Dimensão dos vetores: {modelo_embeddings.get_embedding_dimension()}")

# ── 2. Inicializa o banco vetorial ─────────────────────────
cliente_chroma = chromadb.Client()

try:
    cliente_chroma.delete_collection(name="corpus_saneamento")
    print("🗑️ Coleção anterior removida.")
except:
    pass

colecao = cliente_chroma.create_collection(name="corpus_saneamento")
print("✅ Coleção criada!")

# ── 3. Função de anonimização ──────────────────────────────
def anonimizar_texto(texto):
    """Remove referências à SABESP e telefones do texto."""

    # Remove variações do nome SABESP
    texto = re.sub(r'\bSABESP\b', 'a empresa', texto, flags=re.IGNORECASE)
    texto = re.sub(r'Companhia de Saneamento Básico do Estado de São Paulo',
                   'a empresa', texto, flags=re.IGNORECASE)

    # Remove telefones — formatos variados
    texto = re.sub(r'\(?\d{2}\)?\s?\d{4,5}[-\s]?\d{4}',
                   'nossa central de atendimento', texto)
    texto = re.sub(r'\b0800[\s\-]?\d{3}[\s\-]?\d{4}\b',
                   'nossa central de atendimento', texto)
    texto = re.sub(r'\b195\b', 'nossa central de atendimento', texto)

    return texto

# ── 4. Lê os TXTs, anonimiza e divide em chunks ───────────
pasta_processados = Path("Corpus_Saneamento/dados_processados")
documentos = []
ids = []
metadados = []

for txt in pasta_processados.glob("*.txt"):
    texto = txt.read_text(encoding="utf-8")
    texto = anonimizar_texto(texto)  # anonimiza antes de indexar
    palavras = texto.split()
    chunks = [" ".join(palavras[i:i+500]) for i in range(0, len(palavras), 500)]

    for j, chunk in enumerate(chunks):
        documentos.append(chunk)
        ids.append(f"{txt.stem}_chunk{j}")
        metadados.append({
            "arquivo": txt.name,
            "categoria": txt.stem.split("_")[0],
            "chunk": j
        })

print(f"\n📄 Documentos encontrados: {len(set(m["arquivo"] for m in metadados))}")
print(f"📦 Total de chunks gerados: {len(documentos)}")

# ── 5. Gera embeddings e indexa no ChromaDB ────────────────
print("\n⏳ Gerando embeddings...")
embeddings = modelo_embeddings.encode(documentos, show_progress_bar=True).tolist()

colecao.add(
    documents=documentos,
    embeddings=embeddings,
    ids=ids,
    metadatas=metadados
)

print(f"\n✅ Banco vetorial criado!")
print(f"   Total de chunks indexados: {colecao.count()}")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-monitoring 2.32.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-bigquery-connection 1.23.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-resource-manager 1.19.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-functions 1.25.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
grpc-google-iam-v1 0.14.6 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-storage-control 1.17.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-dataproc 5.32.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompat

/tmp/ipykernel_1109/4063156404.py:3: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


⏳ Carregando modelo de embeddings...


config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  438MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

... (7 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

✅ Modelo carregado! Dimensão dos vetores: 768
✅ Coleção criada!

📄 Documentos encontrados: 48
📦 Total de chunks gerados: 88

⏳ Gerando embeddings...


Batches:   0%|          | 0/3 [00:00<?, ?it/s]


✅ Banco vetorial criado!
   Total de chunks indexados: 88


In [17]:
from groq import Groq

from google.colab import userdata
GROQ_API_KEY = userdata.get("GROQ_API_KEY")  # configure em Secrets (ícone de chave)
cliente_groq = Groq(api_key=GROQ_API_KEY)
print("\n✅ Cliente Groq configurado!")


✅ Cliente Groq configurado!


In [18]:
def rag_saneamento(pergunta, n_resultados=3):
    """
    Pipeline RAG completo com anonimização em duas camadas:
    1. Textos já indexados sem SABESP/telefones
    2. Prompt instrui o modelo a não mencionar esses dados
    """

    # Busca semântica
    embedding_pergunta = modelo_embeddings.encode([pergunta]).tolist()
    resultados = colecao.query(
        query_embeddings=embedding_pergunta,
        n_results=n_resultados
    )

    contexto = "\n\n".join(resultados["documents"][0])
    fontes = [m["arquivo"] for m in resultados["metadatas"][0]]

    # Prompt com instruções de anonimização
    prompt_sistema = """Você é um assistente virtual de atendimento ao cliente
de uma empresa de saneamento básico. Responda de forma clara, objetiva e
educada, com base apenas nas informações fornecidas no contexto abaixo.
Se a informação não estiver no contexto, diga que não possui essa informação
e oriente o usuário a entrar em contato com a empresa pelo canal oficial.

REGRAS IMPORTANTES:
- Nunca mencione o nome "SABESP" nas respostas
- Nunca mencione números de telefone nas respostas
- Quando o contexto sugerir contato telefônico, substitua por:
  "entre em contato com nossa central de atendimento"
- Trate todos os documentos como se fossem da empresa local"""

    prompt_usuario = f"""Contexto:
{contexto}

Pergunta do usuário: {pergunta}

Responda com base no contexto acima."""

    resposta = cliente_groq.chat.completions.create(
        model="qwen/qwen3.8-27b",
        messages=[
            {"role": "system", "content": prompt_sistema},
            {"role": "user", "content": prompt_usuario}
        ],
        temperature=0.3
    )

    return {
        "resposta": resposta.choices[0].message.content,
        "fontes": fontes,
        "contexto_usado": contexto
    }

print("✅ Função RAG definida e pronta!")

✅ Função RAG definida e pronta!


In [19]:
perguntas_teste = [
    "Como solicitar religação do serviço de água?",
    "O que fazer quando há variação no consumo de água?",
    "Quais são as regras para instalação hidráulica?"
]

for pergunta in perguntas_teste:
    print(f"\n{'='*60}")
    print(f"❓ Pergunta: {pergunta}")
    resultado = rag_saneamento(pergunta)
    print(f"💬 Resposta: {resultado['resposta']}")
    print(f"📄 Fontes: {resultado['fontes']}")


❓ Pergunta: Como solicitar religação do serviço de água?
💬 Resposta: Para solicitar a religação do serviço de água após um corte por inadimplência, você deve seguir os seguintes passos:

1. **Quitar o débito**: Pague a(s) conta(s) em atraso que originaram o corte.
2. **Comprovar o pagamento**: Apresente o comprovante de pagamento ou aguarde a compensação bancária, que geralmente ocorre no dia útil seguinte ao pagamento.

Após a confirmação do pagamento, a ordem de religação é gerada e o serviço será restabelecido. O prazo para a religação pode variar conforme a demanda operacional do dia.

Você pode realizar essa solicitação pelos canais de atendimento: WhatsApp, nossa central de atendimento ou presencialmente nas Unidades de Atendimento ao Usuário (UAUs).
📄 Fontes: ['saae_sint_009.txt', 'saae_sint_010.txt', 'saae_sint_008.txt']

❓ Pergunta: O que fazer quando há variação no consumo de água?
💬 Resposta: Com base nas informações fornecidas, quando há variação no consumo de água (geralm

In [20]:
import time

# ── Perguntas de avaliação ─────────────────────────────────────────────────
perguntas_avaliacao = [
    # Cobertas pelo corpus (devem ter boa resposta)
    ("P01", "contas",        "Como solicitar a 2ª via da conta de água?"),
    ("P02", "contas",        "O que é necessário para fazer exclusão de usuário?"),
    ("P03", "contas",        "Como solicitar parcelamento de débito?"),
    ("P04", "contas",        "O que é Tarifa Social e como solicitar?"),
    ("P05", "solicitacoes",  "Como solicitar uma ordem de serviço?"),
    ("P06", "tecnico",       "O que fazer quando há variação no consumo de água?"),
    ("P07", "tecnico",       "Quais são as regras para instalação hidráulica?"),
    ("P08", "regulatorio",   "Quais são os direitos do usuário segundo o regulamento do SAAE?"),
    # Fora do corpus (devem redirecionar para atendimento)
    ("P09", "fora_corpus",   "Qual o horário de funcionamento do SAAE aos sábados?"),
    ("P10", "fora_corpus",   "Como agendar visita técnica para vistoria de vazamento?"),
]

resultados_avaliacao = []

print("=" * 70)
print("AVALIAÇÃO DO SISTEMA RAG")
print("=" * 70)

for pid, categoria, pergunta in perguntas_avaliacao:
    print(f"\n[{pid}] {pergunta}")
    inicio = time.time()

    # Busca semântica com distâncias
    embedding_q = modelo_embeddings.encode([pergunta]).tolist()
    res = colecao.query(
        query_embeddings=embedding_q,
        n_results=3,
        include=["documents", "metadatas", "distances"]
    )

    tempo_recuperacao = time.time() - inicio

    # Geração da resposta
    inicio_geracao = time.time()
    resultado = rag_saneamento(pergunta)
    tempo_total = time.time() - inicio_geracao + tempo_recuperacao

    distancias = res["distances"][0]
    fontes = [m["arquivo"] for m in res["metadatas"][0]]
    categorias_fontes = [m["categoria"] for m in res["metadatas"][0]]

    print(f"  Tempo total  : {tempo_total:.2f}s")
    print(f"  Distâncias   : {[round(d,4) for d in distancias]}")
    print(f"  Fontes       : {fontes}")
    print(f"  Resposta     : {resultado['resposta'][:200]}...")

    resultados_avaliacao.append({
        "id": pid,
        "categoria": categoria,
        "pergunta": pergunta,
        "tempo_total_s": round(tempo_total, 2),
        "dist_chunk1": round(distancias[0], 4),
        "dist_chunk2": round(distancias[1], 4),
        "dist_chunk3": round(distancias[2], 4),
        "fonte_1": fontes[0],
        "fonte_2": fontes[1],
        "fonte_3": fontes[2],
        "cat_fonte_1": categorias_fontes[0],
        "resposta_completa": resultado["resposta"]
    })

# ── Resumo quantitativo ────────────────────────────────────────────────────
import statistics

tempos = [r["tempo_total_s"] for r in resultados_avaliacao]
dist1  = [r["dist_chunk1"]   for r in resultados_avaliacao]

print("\n" + "=" * 70)
print("RESUMO QUANTITATIVO")
print("=" * 70)
print(f"Perguntas avaliadas       : {len(resultados_avaliacao)}")
print(f"Tempo médio de resposta   : {statistics.mean(tempos):.2f}s")
print(f"Tempo mínimo              : {min(tempos):.2f}s")
print(f"Tempo máximo              : {max(tempos):.2f}s")
print(f"Distância coseno média    : {statistics.mean(dist1):.4f}")
print(f"Distância coseno mínima   : {min(dist1):.4f}  (chunk mais relevante)")
print(f"Distância coseno máxima   : {max(dist1):.4f}  (chunk menos relevante)")

# ── Salva resultados em CSV ────────────────────────────────────────────────
import pandas as pd
df_aval = pd.DataFrame(resultados_avaliacao)
caminho_aval = raiz / "resultados" / "avaliacao_rag.csv"
df_aval.to_csv(caminho_aval, index=False, encoding="utf-8")
print(f"\n💾 Resultados salvos em: {caminho_aval}")

AVALIAÇÃO DO SISTEMA RAG

[P01] Como solicitar a 2ª via da conta de água?
  Tempo total  : 1.60s
  Distâncias   : [33.351, 34.5645, 35.0014]
  Fontes       : ['saae_sint_010.txt', 'saae_sint_009.txt', 'saae_sint_002.txt']
  Resposta     : Para solicitar a 2ª via da conta de água, você pode utilizar um dos seguintes canais:

1. **Presencialmente**: Dirija-se a qualquer Unidade de Atendimento ao Usuário (UAU) apresentando um documento de...

[P02] O que é necessário para fazer exclusão de usuário?
  Tempo total  : 0.88s
  Distâncias   : [32.6198, 35.816, 36.4149]
  Fontes       : ['saae_sint_002.txt', 'saae_sint_009.txt', 'saae_sint_010.txt']
  Resposta     : Não possuo informações sobre o procedimento para exclusão de usuário no contexto fornecido. Para obter essa orientação, recomendo que você entre em contato com nossa central de atendimento....

[P03] Como solicitar parcelamento de débito?
  Tempo total  : 1.27s
  Distâncias   : [34.6521, 37.5197, 40.4626]
  Fontes       : ['saae_sin

In [ ]:
import os
from pathlib import Path

pasta = raiz / "dados_processados"
arquivos = list(pasta.glob("*.txt"))

categorias = {
    "historico": [f for f in arquivos if f.name.startswith("historico")],
    "sintetico": [f for f in arquivos if f.name.startswith("saae_sint")],
    "outros": [f for f in arquivos if not f.name.startswith("historico") and not f.name.startswith("saae_sint")],
}

for cat, files in categorias.items():
    print(f"{cat}: {len(files)} arquivo(s)")
    for f in files:
        print(f"  {f.name}")